In [14]:
!nvidia-smi

Sat Aug 22 19:46:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             32W /   70W |    1621MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
!pip install transformer_lens

In [16]:
import torch
from transformer_lens import HookedTransformer

In [17]:
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

SEED = 42
torch.manual_seed(SEED)

PyTorch: 2.11.0+cu128
CUDA available: True
Running on: cuda


In [18]:
model = HookedTransformer.from_pretrained("gpt2", device=device)
model.eval()
print(f"Model loaded. Layers: {model.cfg.n_layers}, d_model: {model.cfg.d_model}")

/tmp/ipykernel_1272/3268298616.py:1: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2", device=device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Model loaded. Layers: 12, d_model: 768


In [19]:
# One generation — confirm English output
PROMPT = "The capital of France is"

tokens = model.to_tokens(PROMPT)
with torch.no_grad():
    output = model.generate(tokens, max_new_tokens=5, do_sample=False)

print("Prompt:", PROMPT)
print("Completion:", model.to_string(output[0]))

  0%|          | 0/5 [00:00<?, ?it/s]

Prompt: The capital of France is
Completion: <|endoftext|>The capital of France is now home to the world


In [20]:
with torch.no_grad():
    logits, cache = model.run_with_cache(PROMPT)

shape = cache["resid_post", 6].shape
print(f"cache['resid_post', 6].shape: {shape}")

cache['resid_post', 6].shape: torch.Size([1, 6, 768])


In [21]:
# Inspect the token positions
token_strs = model.to_str_tokens(PROMPT)
print("Tokens:", list(enumerate(token_strs)))
print()

batch, pos, d_model = shape

print(f"batch   = {batch}")
print(f"  -> number of input sequences. 1 because we passed a single string.")
print(f"  -> in extract.py we'll loop pairs and process each prompt separately.")
print()
print(f"pos     = {pos}")
print(f"  -> number of token positions in the input: {token_strs}")
print(f"  -> each position gets its own {d_model}-d residual stream vector.")
print(f"  -> for our reading vectors we'll take the FINAL token position (index -1)")
print(f"  -> because that's where the model 'decides' how to continue the reasoning.")
print()
print(f"d_model = {d_model}")
print(f"  -> width of the residual stream (GPT-2 small is 768).")
print(f"  -> the reading vector we extract will be a direction in this space.")
print(f"  -> diff-of-means: v_L = mean(misaligned_acts) - mean(aligned_acts), shape [{d_model}].")

Tokens: [(0, '<|endoftext|>'), (1, 'The'), (2, ' capital'), (3, ' of'), (4, ' France'), (5, ' is')]

batch   = 1
  -> number of input sequences. 1 because we passed a single string.
  -> in extract.py we'll loop pairs and process each prompt separately.

pos     = 6
  -> number of token positions in the input: ['<|endoftext|>', 'The', ' capital', ' of', ' France', ' is']
  -> each position gets its own 768-d residual stream vector.
  -> for our reading vectors we'll take the FINAL token position (index -1)
  -> because that's where the model 'decides' how to continue the reasoning.

d_model = 768
  -> width of the residual stream (GPT-2 small is 768).
  -> the reading vector we extract will be a direction in this space.
  -> diff-of-means: v_L = mean(misaligned_acts) - mean(aligned_acts), shape [768].


In [22]:
assert len(shape) == 3, "shape should be 3-dimensional"
assert shape[0] == 1,   "batch should be 1 for a single string input"
assert shape[2] == 768, "d_model should be 768 for GPT-2 small"

print("✓ Setup complete. Toolchain is live.")
print(f"  cache['resid_post', 6].shape = {tuple(shape)}")


✓ Setup complete. Toolchain is live.
  cache['resid_post', 6].shape = (1, 6, 768)
